# 17. 🚀 Продвинутое: ядра, гауссовские процессы, оптимальный транспорт

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/17_kernels_gp_ot.ipynb)

Код из раздела [modules/17_kernels_gp_ot.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/17_kernels_gp_ot.md#m17). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Оптимальный транспорт

In [ ]:
import numpy as np
def sinkhorn(a, b, C, eps=0.05, iters=500):
    """a, b — веса точек (суммы 1), C — матрица стоимостей. Возвращает транспортный план."""
    K = np.exp(-C / eps); u = np.ones_like(a)
    for _ in range(iters):
        v = b / (K.T @ u); u = a / (K @ v)
    return u[:, None] * K * v[None, :]

x = np.linspace(0, 1, 50); y = np.linspace(0.3, 1.3, 50)
C = (x[:, None] - y[None, :]) ** 2
P = sinkhorn(np.full(50, 1 / 50), np.full(50, 1 / 50), C)
print(np.sum(P * C))       # ≈ 0.11: W₂² = 0.3² = 0.09 плюс «размытие» плана из-за энтропии (меньше eps → ближе к 0.09)

### 🏋️ Практика модуля

**17.1.** Проверьте, что матрица Грама RBF-ядра на случайных точках положительно полуопределена, а «ядро» $k(x, x') = -\lVert x - x'\rVert$ — нет.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
X = np.random.randn(30, 2)
D = np.linalg.norm(X[:, None] - X[None], axis=-1)
print(np.linalg.eigvalsh(np.exp(-D**2 / 2)).min())   # ≥ 0 (с точностью до округления)
print(np.linalg.eigvalsh(-D).min())                  # < 0 — не ядро

### 🏋️ Практика модуля

**17.2.** Реализуйте предсказание GP и проверьте, что среднее проходит через точки данных при малом шуме.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
k = lambda a, b: np.exp(-0.5 * (a[:, None] - b[None, :]) ** 2)
Xtr = np.array([-2., 0., 1.5]); ytr = np.sin(Xtr)
K = k(Xtr, Xtr) + 1e-6 * np.eye(3)
mu = k(Xtr, Xtr) @ np.linalg.solve(K, ytr)
print(np.allclose(mu, ytr, atol=1e-4))               # True

### 🏋️ Практика модуля

**17.3.** Найдите $W_1$ между выборками из $\mathcal{N}(0, 1)$ и $\mathcal{N}(2, 1)$ через отсортированные значения.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
a = np.sort(np.random.randn(100_000)); b = np.sort(np.random.randn(100_000) + 2)
print(np.mean(np.abs(a - b)))   # ≈ 2.0 — сдвиг среднего

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def kernel_trick():
    rng = np.random.default_rng(2)
    n = 200
    r = np.r_[rng.uniform(0, 1, n), rng.uniform(1.6, 2.4, n)]
    t = rng.uniform(0, 2 * np.pi, 2 * n)
    X = np.c_[r * np.cos(t), r * np.sin(t)]; y = np.r_[np.zeros(n), np.ones(n)]
    fig = plt.figure(figsize=(12, 4.6))
    ax = fig.add_subplot(1, 2, 1)
    ax.scatter(*X[y == 0].T, s=10, color=C[0]); ax.scatter(*X[y == 1].T, s=10, color=C[1])
    ax.set_aspect("equal"); ax.set_title("В исходном 2D прямой не разделить")
    ax = fig.add_subplot(1, 2, 2, projection="3d")
    z = (X**2).sum(1)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], z[y == 0], s=8, color=C[0])
    ax.scatter(X[y == 1, 0], X[y == 1, 1], z[y == 1], s=8, color=C[1])
    G = np.linspace(-2.5, 2.5, 10); GX, GY = np.meshgrid(G, G)
    ax.plot_surface(GX, GY, np.full_like(GX, 1.7), alpha=0.25, color=C[2])
    ax.set_title("φ(x) = (x₁, x₂, x₁² + x₂²): разделяет плоскость")
    save(fig, "kernel_trick")

kernel_trick()

In [ ]:
def gaussian_process():
    rng = np.random.default_rng(4)
    k = lambda a, b, l=0.8: np.exp(-0.5 * (a[:, None] - b[None, :]) ** 2 / l**2)
    Xtr = np.array([-3.5, -2.0, -0.5, 1.0, 2.5]); ytr = np.sin(Xtr) + 0.05 * rng.normal(size=5)
    xs = np.linspace(-5, 5, 300)
    K = k(Xtr, Xtr) + 1e-3 * np.eye(5); Ks = k(xs, Xtr); Kss = k(xs, xs)
    mu = Ks @ np.linalg.solve(K, ytr); cov = Kss - Ks @ np.linalg.solve(K, Ks.T)
    sd = np.sqrt(np.clip(np.diag(cov), 0, None))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    prior = rng.multivariate_normal(np.zeros(300), Kss + 1e-6 * np.eye(300), 4)
    for p, c in zip(prior, C): axes[0].plot(xs, p, color=c, lw=1.5)
    axes[0].set_title("Априор: случайные гладкие функции из ядра RBF")
    axes[1].fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=C[0], alpha=0.2, label="±2σ")
    post = rng.multivariate_normal(mu, cov + 1e-6 * np.eye(300), 3)
    for p in post: axes[1].plot(xs, p, color=C[0], lw=0.8, alpha=0.6)
    axes[1].plot(xs, mu, color=C[0], lw=2.5, label="среднее"); axes[1].plot(xs, np.sin(xs), "k--", lw=1, label="истина")
    axes[1].scatter(Xtr, ytr, color=C[1], zorder=5, s=50, label="данные")
    axes[1].legend(fontsize=8, loc="lower left"); axes[1].set_title("Апостериор: неопределённость растёт вдали от данных")
    save(fig, "gaussian_process")

gaussian_process()

In [ ]:
def wasserstein():
    shifts = np.linspace(0, 6, 61)
    x = np.linspace(-10, 16, 4000); dx = x[1] - x[0]
    P = stats.norm.pdf(x, 0, 0.5)
    kl, js, w1 = [], [], []
    for s in shifts:
        Q = stats.norm.pdf(x, s, 0.5)
        kl.append(np.sum(P * np.log((P + 1e-300) / (Q + 1e-300))) * dx)
        M = 0.5 * (P + Q)
        js.append(0.5 * np.sum(P * np.log((P + 1e-300) / (M + 1e-300))) * dx + 0.5 * np.sum(Q * np.log((Q + 1e-300) / (M + 1e-300))) * dx)
        w1.append(s)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    axes[0].fill_between(x, P, alpha=0.4, color=C[0], label="P = N(0, 0.5²)")
    axes[0].fill_between(x, stats.norm.pdf(x, 4, 0.5), alpha=0.4, color=C[1], label="Q = N(θ, 0.5²), θ = 4")
    axes[0].annotate("", xy=(4, 0.5), xytext=(0, 0.5), arrowprops=dict(arrowstyle="->", lw=2))
    axes[0].text(1.2, 0.56, "перевезти массу на θ"); axes[0].set_xlim(-3, 7); axes[0].legend(fontsize=9)
    axes[0].set_title("Оптимальный транспорт: «сколько работы» сдвинуть P в Q")
    axes[1].plot(shifts, w1, color=C[2], lw=2.5, label="W₁ = |θ| — гладко растёт")
    axes[1].plot(shifts, js, color=C[1], lw=2.5, label="JS → log 2: градиент ≈ 0")
    axes[1].plot(shifts, np.minimum(kl, 6), color=C[0], lw=1.5, ls="--", label="KL (обрезано): быстро растёт")
    axes[1].set_xlabel("сдвиг θ"); axes[1].legend(fontsize=9)
    axes[1].set_title("Почему WGAN: расстояние Вассерштейна даёт полезный градиент")
    save(fig, "wasserstein")

wasserstein()